In [3]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

def load_input(file_path):
    data = pd.read_csv(file_path, header=None, names=['x1', 'x2', 'label'])
    features = data[['x1', 'x2']].values
    targets = data['label'].values
    return features, targets

def heuristic_perceptron(features, labels, lr, num_epochs):
    weights = np.random.randn(features.shape[1])
    bias = 0.0
    history = [(weights.copy(), bias)]

    for _ in range(num_epochs):
        for xi, yi in zip(features, labels):
            activation = np.dot(weights, xi) + bias
            prediction = 1 if activation > 0 else 0
            update = lr * (yi - prediction)
            weights += update * xi
            bias += update
        history.append((weights.copy(), bias))

    return history

def line_from_weights(w, x_vals):
    weights, bias = w
    if abs(weights[1]) < 1e-6:
        return np.full_like(x_vals, -bias / weights[0])
    return -(weights[0] * x_vals + bias) / weights[1]

def show_learning_steps(track, points, labels, plot_title):
    fig = go.Figure()

    for cls, clr in zip([0, 1], ['royalblue', 'crimson']):
        fig.add_trace(go.Scatter(
            x=points[labels == cls, 0],
            y=points[labels == cls, 1],
            mode='markers',
            marker=dict(color=clr, size=10, line=dict(width=1, color='black')),
            name=f'Class {cls}',
            hoverinfo='x+y+name'
        ))

    x_min, x_max = points[:, 0].min() - 0.1, points[:, 0].max() + 0.1
    x_vals = np.array([x_min, x_max])

    for idx, state in enumerate(track):
        y_vals = line_from_weights(state, x_vals)

        if idx == 0:
            color, dash, width, label = 'red', 'solid', 3, 'Initial Boundary'
        elif idx == len(track) - 1:
            color, dash, width, label = 'black', 'solid', 3, 'Final Boundary'
        else:
            color, dash, width, label = 'green', 'dash', 1, None

        fig.add_trace(go.Scatter(
            x=x_vals, y=y_vals,
            mode='lines',
            line=dict(color=color, dash=dash, width=width),
            name=label,
            showlegend=(label is not None)
        ))

    fig.update_layout(
        title=dict(text=plot_title, font=dict(size=20, family="Arial Black")),
        xaxis=dict(title='x₁', gridcolor='lightgray', zeroline=False),
        yaxis=dict(title='x₂', gridcolor='lightgray', zeroline=False),
        plot_bgcolor='white',
        font=dict(size=14),
        legend=dict(
            bgcolor='rgba(255,255,255,0.8)',
            bordercolor='gray',
            borderwidth=1
        ),
        margin=dict(l=40, r=40, t=60, b=40)
    )
    fig.show()

# Run Part 1
if __name__ == "__main__":
    np.random.seed(0)
    X, y = load_input('/content/data.csv')

    for epochs in [15, 20, 40, 80, 95]:
        for rate in [0.01, 0.1, 1.0]:
            path = heuristic_perceptron(X, y, rate, epochs)
            show_learning_steps(path, X, y, f"Heuristic Perceptron (α={rate}, Epochs={epochs})")


In [5]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def read_data(filename):
    df = pd.read_csv(filename, header=None, names=['f1', 'f2', 'out'])
    return df[['f1', 'f2']].values, df['out'].values

def sigmoid_fn(z):
    return 1 / (1 + np.exp(-z))

def train_sigmoid_perceptron(X, y, lr, epochs):
    w = np.random.randn(X.shape[1])
    b = 0.0
    states = [(w.copy(), b)]
    losses = []

    for _ in range(epochs):
        for xi, yi in zip(X, y):
            z = np.dot(w, xi) + b
            pred = sigmoid_fn(z)
            grad = yi - pred
            w += lr * grad * xi
            b += lr * grad
        states.append((w.copy(), b))

        z_all = np.dot(X, w) + b
        predictions = sigmoid_fn(z_all)
        log_loss = -np.mean(y * np.log(predictions + 1e-9) + (1 - y) * np.log(1 - predictions + 1e-9))
        losses.append(log_loss)

    return states, losses

def visualize_all(X, y, rates, epoch_list):
    for epoch in epoch_list:
        fig = make_subplots(rows=2, cols=len(rates),
                            subplot_titles=[f"Boundary η={r}, E={epoch}" for r in rates] +
                                             [f"Loss η={r}" for r in rates])

        for idx, lr in enumerate(rates):
            steps, log_losses = train_sigmoid_perceptron(X, y, lr, epoch)

            for cls, color in zip([0, 1], ['blue', 'red']):
                mask = (y == cls)
                fig.add_trace(go.Scatter(
                    x=X[mask, 0], y=X[mask, 1],
                    mode='markers',
                    marker=dict(color=color, size=9, line=dict(width=1, color='black')),
                    name=f'Class {cls}' if idx == 0 else '',
                    showlegend=(idx == 0)
                ), row=1, col=idx + 1)

            for step_idx, (w, b) in enumerate(steps):
                x_vals = np.array([0.0, 1.0])
                if abs(w[1]) < 1e-6:
                    y_vals = [0.0, 1.0]
                else:
                    y_vals = [-(w[0] * x + b) / w[1] for x in x_vals]

                color, dash, width, name = 'green', 'dash', 1, None
                if step_idx == 0:
                    color, dash, width, name = 'red', 'solid', 3, 'Start'
                elif step_idx == len(steps) - 1:
                    color, dash, width, name = 'black', 'solid', 3, 'Final'

                fig.add_trace(go.Scatter(
                    x=x_vals, y=y_vals, mode='lines',
                    line=dict(color=color, dash=dash, width=width),
                    name=name,
                    showlegend=(name is not None and idx == 0)
                ), row=1, col=idx + 1)

            fig.add_trace(go.Scatter(
                x=list(range(1, len(log_losses) + 1)), y=log_losses,
                mode='lines+markers', showlegend=False,
                line=dict(color='purple')
            ), row=2, col=idx + 1)

            fig.update_xaxes(title_text='Epoch', row=2, col=idx + 1)
            fig.update_yaxes(title_text='Log Loss', row=2, col=idx + 1)

        fig.update_layout(
            title=dict(text=f"Gradient Descent Perceptron – Epochs = {epoch}", font=dict(size=22, family='Arial Black')),
            plot_bgcolor='white',
            height=850,
            width=1450,
            font=dict(size=14),
            legend=dict(
                bgcolor='rgba(255,255,255,0.85)',
                bordercolor='gray',
                borderwidth=1
            ),
            margin=dict(l=40, r=40, t=70, b=40)
        )
        fig.show()

# Run Part 2
if __name__ == "__main__":
    np.random.seed(42)
    features, targets = read_data('/content/data.csv')
    learning_rates = [0.01, 0.1, 1.0]
    all_epochs = [100, 150, 180, 200]
    visualize_all(features, targets, learning_rates, all_epochs)
